In [14]:
import sys
sys.path.append('../')

import numpy as np
from matplotlib import pyplot as plt
from qutip.qip.operations import rz, cz_gate
from tqdm import tqdm
from matplotlib.colors import LogNorm
import pytz, cmath, itertools
import scqubits.settings as settings
settings.OVERLAP_THRESHOLD = 0.3
from joblib import Parallel, delayed
import scipy.sparse as ssp
from sympy import symbols
import utils_2Q_gate_zp as ut
import pandas as pd
import scipy as sp
from multiprocessing import Pool
import qutip as qt
import multiprocessing as mp
from multiprocessing import Pool
import scqubits as scq
from sympy import symbols
import scipy.sparse as ssp
from datetime import datetime


In [15]:
truc_import = 500
truc1, truc_tot, charge_pick = 10, 10, False

folder = f'../../data/3ncut_two_zeropi/truc1={truc_import}/'
eval0 = pd.read_csv(folder+ 'eval0.txt').to_numpy().flatten()
eval1 = pd.read_csv(folder+ 'eval1.txt').to_numpy().flatten()
n_theta0 = pd.read_csv(folder+ 'n_theta0.txt').map(complex).to_numpy()
n_theta1 = pd.read_csv(folder+ 'n_theta1.txt').map(complex).to_numpy()

eval0 = eval0[:truc1]
eval1 = eval1[:truc1]
hspace_0 = np.arange(truc1)
hspace_1 = np.arange(truc1)
n_theta0 = qt.Qobj(n_theta0[np.ix_(hspace_0, hspace_0)])
n_theta1 = qt.Qobj(n_theta1[np.ix_(hspace_1, hspace_1)])

In [16]:
zp = scq.Circuit(ut.zp_yml, from_file=False)
zp.configure(transformation_matrix=np.linalg.inv(ut.transform_2zeropi))
system_hierarchy = [[1,2],  [5,6]]
subsystem_trunc_dims = [100, 100]
zp.configure(system_hierarchy=system_hierarchy,
            subsystem_trunc_dims=subsystem_trunc_dims
            )
# zp.cutoff_ext_1, zp.cutoff_ext_5 = 300, 300
# zp.cutoff_n_2, zp.cutoff_n_6 = 90, 90
zp.cutoff_ext_1, zp.cutoff_ext_5 = 50, 50
zp.cutoff_n_2, zp.cutoff_n_6 = 20, 20
##############################################################################################
###  Compute eigenvalues and eigenvectors for coupling H
n2, n6 = symbols('n2 n6')
g = float(zp.sym_interaction((1,0), return_expr=True).coeff(n2*n6) )
Hint = qt.tensor(qt.Qobj(n_theta0) , qt.Qobj(n_theta1))
H_bare = (  qt.tensor(qt.Qobj(np.diag(eval0)),  qt.identity(len(hspace_1)))
        +  qt.tensor(qt.identity(len(hspace_0)),  qt.Qobj(np.diag(eval1))) )
# Htot = (g* Hint + H_bare).tidyup(atol=1e-8)
Htot = g* Hint + H_bare


In [17]:

### the one-line code below takes time when truc1 is large
k = Htot.shape[0] - 1
if truc_tot != None:
    k = truc_tot
eval_tot, eket_tot = ssp.linalg.eigsh(Htot.data, k=k, which='SA', tol=1.e-10)

sorted_idx_tot = np.argsort(eval_tot)
eval_tot = eval_tot[sorted_idx_tot]
eval_tot = eval_tot - eval_tot[0]
eket_tot = ssp.csr_matrix([eket_tot[:,idx] for idx in sorted_idx_tot])

In [18]:
eval_tot

array([0.        , 2.40936448, 2.5955978 , 3.43467329, 3.43599989,
       3.46859009, 3.46923875, 4.69761836, 4.95008166, 5.09656943])

In [20]:
qt.Qobj(eket_tot)

Quantum object: dims = [[10], [100]], shape = (10, 100), type = oper, isherm = False
Qobj data =
[[-3.40352525e-01-9.40233731e-01j -1.33601171e-16+1.79606029e-17j
  -7.53925870e-07-2.08274210e-06j -1.21384441e-17+1.14641006e-16j
   6.10357190e-05+1.68612946e-04j -1.82063898e-17+3.90755448e-17j
  -7.32337192e-18+1.66086008e-17j  2.76282986e-17-2.58212102e-18j
  -6.45117596e-17+5.49397985e-17j  1.95746207e-06+5.40754581e-06j
  -6.61615841e-17-9.69459408e-17j -3.73688828e-03-1.03232624e-02j
   2.63294611e-17-1.96189980e-17j -1.75833026e-17-2.30877307e-17j
   4.65126129e-17+1.68331118e-17j -1.30979214e-05-3.61833882e-05j
  -4.25027607e-18-6.31115701e-17j  1.50184282e-17-1.42042722e-17j
   7.09579474e-05+1.96023390e-04j  3.08540110e-17-2.85366176e-17j
  -3.89646349e-07-1.07640935e-06j  2.80452082e-18-3.39272296e-18j
  -2.47520167e-09-6.83781620e-09j -1.70296803e-17+2.11336732e-17j
   2.31230250e-07+6.38780280e-07j -3.34519514e-17-6.22691166e-18j
   6.65750851e-18+3.37656676e-17j -2.37846822

In [21]:
=

SyntaxError: invalid syntax (1763773627.py, line 1)

In [24]:
truc1, truc_tot, charge_pick = 10, 11, False
folder = f'data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()
eval_tot = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
n_theta0_dress = 2*np.pi* pd.read_csv(folder+ 'n_theta0_dress.txt').map(complex).to_numpy()
n_theta1_dress = 2*np.pi* pd.read_csv(folder+ 'n_theta1_dress.txt').map(complex).to_numpy()
eket_tot = pd.read_csv(folder+ 'eket_tot.txt').map(complex).to_numpy()

In [12]:
eval_tot/(2*np.pi)
hspace_full

['0-0', '0-1', '1-0', '0-2', '0-3', '2-0', '3-0', '0-4', '4-0', '1-1']

In [25]:
qt.Qobj(n_theta1_dress/(2*np.pi))
# qt.Qobj(eket_tot)


Quantum object: dims = [[11], [11]], shape = (11, 11), type = oper, isherm = True
Qobj data =
[[ 2.98641108e-15+0.00000000e+00j  6.60420761e-01+1.06383970e+00j
  -3.31160058e-01+2.12771759e-01j -2.65932095e-16+9.72595962e-17j
   7.69770507e-16-4.10430348e-16j -7.76872499e-16+2.92522853e-16j
   1.14689124e-15-4.56048710e-16j -4.15577008e-16+1.48031368e-15j
   5.79669444e-15+4.02604090e-16j -1.61294713e-15+5.61474234e-15j
   5.57618947e-03-4.84853597e-03j]
 [ 6.60420761e-01-1.06383970e+00j -1.91177720e-16-6.16297582e-33j
  -1.72458070e-17-7.30266637e-16j -1.77450445e-02-3.42176858e-03j
  -1.19208123e-14-3.12376322e-15j  1.24762390e-04+1.45335171e-04j
  -9.86558618e-16-4.34215191e-16j  1.92894620e-01+1.74903048e+00j
   8.55405360e-02+8.44263912e-03j  1.33333596e-02-4.99208765e-02j
   3.27880519e-14+8.82592954e-14j]
 [-3.31160058e-01-2.12771759e-01j -1.72458070e-17+7.30266637e-16j
   1.31189502e-15-1.10933565e-31j -1.09059308e-03+5.21869543e-03j
  -7.62770469e-16+2.42583586e-15j -3.7926155